# Module 21: Hands-On Lab — UDFs & Real-World Patterns

**Snowpark ML Fundamentals — Day 2 Afternoon Lab**

---

## Scenario

You’re building feature engineering UDFs and a scoring pipeline
for NCL’s guest analytics platform. Apply the patterns from Module 20.

## Rules
- Fill in every `# TODO:` — each requires **1–5 lines of code**
- Run the **Validation** cells to check your work
- Estimated time: **30 minutes**

---

In [ ]:
# --- Setup (given — just run this cell) ---
import sys
sys.path.insert(0, '../../../src')

import logging
import warnings

import pandas as pd
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("snowflake.snowpark").setLevel(logging.ERROR)
logging.getLogger("snowflake.ml").setLevel(logging.ERROR)

logger = logging.getLogger("module_21")
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(levelname)s:%(name)s:%(message)s"))
    logger.addHandler(handler)
logger.setLevel(logging.INFO)
logger.propagate = False

from snowflake.snowpark import functions as F
from snowflake.snowpark import Window
from snowflake.snowpark.types import IntegerType, FloatType, StringType
from snowflake.snowpark.functions import pandas_udf

from snowpark_fundamentals.session import create_session
from snowpark_fundamentals.data.sample_data import create_sample_orders_dataset

session = create_session(env_path='../../../.env')

# Schema isolation
STUDENT_NAME = "YOUR_NAME"  # <-- CHANGE THIS
session.sql("USE ROLE MLDS_ROLE_D").collect()
session.sql("USE DATABASE MLDS_D").collect()
session.sql(f"CREATE SCHEMA IF NOT EXISTS {STUDENT_NAME}_ONSITE_TRAINING").collect()
session.sql(f"USE SCHEMA {STUDENT_NAME}_ONSITE_TRAINING").collect()

# Generate data
orders_df = create_sample_orders_dataset(session, n_rows=5000)
logger.info(f"Orders: {orders_df.count()} rows")
orders_df.show(5)

---

## Exercise A: Vectorized UDF — Guest Health Score (10 min)

Create a vectorized UDF that computes a **guest health score** (0–100)
combining three signals:

| Signal | Formula | Weight |
|--------|---------|--------|
| Frequency | `min(1.0, order_count / 10.0)` | 40% |
| Monetary | `min(1.0, total_spend / 5000.0)` | 40% |
| Recency | `max(0.0, 1.0 - days_since / 365.0)` | 20% |

**Final score:** `min(100, (frequency * 40) + (monetary * 40) + (recency * 20))`

In [ ]:
# --- Prepare per-customer aggregates for the UDF ---
customer_stats = orders_df.group_by('CUSTOMER_ID').agg(
    F.count('ORDER_ID').alias('ORDER_COUNT'),
    F.sum('ORDER_TOTAL').alias('TOTAL_SPEND'),
    F.datediff('day', F.max('ORDER_DATE'), F.current_date()).alias('DAYS_SINCE_LAST'),
)
customer_stats.show(5)

In [ ]:
# TODO A1: Define the guest_health_score vectorized UDF
# Input: 3 pd.Series (order_count, total_spend, days_since_last)
# Output: pd.Series of float scores (0-100)
# Hint: Follow the @pandas_udf pattern from notebook 20 section 20.2

@pandas_udf(
    name=f"{STUDENT_NAME}_GUEST_HEALTH_SCORE",
    return_type=FloatType(),
    input_types=[IntegerType(), FloatType(), IntegerType()],
    is_permanent=False,
    replace=True,
    packages=['pandas'],
    session=session,
)
def guest_health_score(
    order_count: pd.Series,
    total_spend: pd.Series,
    days_since_last: pd.Series,
) -> pd.Series:
    ____

In [ ]:
# TODO A2: Apply the UDF to customer_stats and display results
# Hint: Use .with_column() and pass the three columns as arguments

result_a = customer_stats.with_column(
    'HEALTH_SCORE',
    ____
)
result_a.sort(F.col('HEALTH_SCORE').desc()).show(10)

In [ ]:
# --- Validation A ---
assert 'HEALTH_SCORE' in result_a.columns, "A2: HEALTH_SCORE column should exist"
sample = result_a.select('HEALTH_SCORE').limit(50).to_pandas()
assert (sample['HEALTH_SCORE'] >= 0).all(), "A1: scores should be >= 0"
assert (sample['HEALTH_SCORE'] <= 100).all(), "A1: scores should be <= 100"
logger.info("Exercise A: All validations passed!")

---

## Exercise B: Window Functions — Time-Series Features (10 min)

Create three window-based features for customer order analysis:

| Feature | Description |
|---------|------------|
| `CUMULATIVE_SPEND` | Running total of ORDER_TOTAL per customer, ordered by date |
| `ORDER_RANK` | Rank of each order within customer's history by ORDER_TOTAL (highest = 1) |
| `MOVING_AVG_3` | 3-order moving average of ORDER_TOTAL per customer |

In [ ]:
# TODO B1: Compute cumulative spend per customer ordered by date
# Hint: Use F.sum('ORDER_TOTAL').over(Window.partition_by(...).order_by(...))

customer_window = Window.partition_by('CUSTOMER_ID').order_by('ORDER_DATE')

result_b = orders_df.with_column(
    'CUMULATIVE_SPEND',
    ____
)
result_b.select('CUSTOMER_ID', 'ORDER_DATE', 'ORDER_TOTAL', 'CUMULATIVE_SPEND') \
    .filter(F.col('CUSTOMER_ID') == 1).show(5)

In [ ]:
# TODO B2: Rank each order within customer's history by ORDER_TOTAL (highest first)
# Hint: Use F.rank().over(Window.partition_by(...).order_by(F.col(...).desc()))

rank_window = Window.partition_by('CUSTOMER_ID').order_by(F.col('ORDER_TOTAL').desc())

result_b = result_b.with_column(
    'ORDER_RANK',
    ____
)
result_b.select('CUSTOMER_ID', 'ORDER_TOTAL', 'ORDER_RANK') \
    .filter(F.col('CUSTOMER_ID') == 1).show(5)

In [ ]:
# TODO B3: Compute 3-order moving average per customer
# Hint: Use F.avg('ORDER_TOTAL').over(Window...rows_between(-2, Window.CURRENT_ROW))

moving_window = (
    Window.partition_by('CUSTOMER_ID')
    .order_by('ORDER_DATE')
    .rows_between(-2, Window.CURRENT_ROW)
)

result_b = result_b.with_column(
    'MOVING_AVG_3',
    ____
)
result_b.select('CUSTOMER_ID', 'ORDER_DATE', 'ORDER_TOTAL', 'MOVING_AVG_3') \
    .filter(F.col('CUSTOMER_ID') == 1).show(5)

In [ ]:
# --- Validation B ---
for col_name in ['CUMULATIVE_SPEND', 'ORDER_RANK', 'MOVING_AVG_3']:
    assert col_name in result_b.columns, f"B: {col_name} column should exist"
sample = result_b.filter(F.col('CUSTOMER_ID') == 1) \
    .sort('ORDER_DATE') \
    .select('CUMULATIVE_SPEND').to_pandas()
# Cumulative spend should be non-decreasing
diffs = sample['CUMULATIVE_SPEND'].diff().dropna()
assert (diffs >= 0).all(), "B1: cumulative spend should be non-decreasing"
logger.info("Exercise B: All validations passed!")

---

## Exercise C: Stored Procedure — Scoring Pipeline (8 min)

Complete the stored procedure below. Three lines inside the handler
need to be filled in.

In [ ]:
# TODO C1/C2/C3: Complete the stored procedure handler
# The procedure reads orders, categorizes them, and writes to an output table.
# Fill in the three ____ lines inside the handler.

scoring_sp = f"""
CREATE OR REPLACE PROCEDURE {STUDENT_NAME}_LAB_SCORE_ORDERS(
    INPUT_TABLE VARCHAR,
    OUTPUT_TABLE VARCHAR
)
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'run'
AS $$
def run(session, input_table: str, output_table: str) -> str:
    from snowflake.snowpark import functions as F

    # TODO C1: Read the input table into a DataFrame
    # Hint: session.table(input_table)
    df = ____

    # Categorize orders
    scored = df.with_column(
        'PRIORITY',
        F.when(F.col('ORDER_TOTAL') > 500, F.lit('URGENT'))
         .when(F.col('ORDER_TOTAL') > 200, F.lit('HIGH'))
         .when(F.col('ORDER_TOTAL') > 50, F.lit('NORMAL'))
         .otherwise(F.lit('LOW'))
    )

    # TODO C2: Write scored DataFrame to the output table (overwrite mode)
    # Hint: scored.write.mode('overwrite').save_as_table(output_table)
    ____

    # TODO C3: Return a success message with the row count
    # Hint: count = session.table(output_table).count()
    count = ____
    return f'Scored {{count}} orders into {{output_table}}'
$$;
"""

session.sql(scoring_sp).collect()
logger.info("Stored procedure created.")

In [ ]:
# Call the stored procedure
result = session.sql(f"""
    CALL {STUDENT_NAME}_LAB_SCORE_ORDERS('SAMPLE_ORDERS', '{STUDENT_NAME}_SCORED_ORDERS')
""").collect()
logger.info(f"Result: {result[0][0]}")

# Check output
session.table(f'{STUDENT_NAME}_SCORED_ORDERS').show(5)

In [ ]:
# --- Validation C ---
output_table = f'{STUDENT_NAME}_SCORED_ORDERS'
scored_count = session.table(output_table).count()
assert scored_count > 0, "C: output table should have rows"
scored_cols = session.table(output_table).columns
assert 'PRIORITY' in scored_cols, "C: PRIORITY column should exist"
logger.info(f"Exercise C: All validations passed! ({scored_count} orders scored)")

---

## Lab Complete!

You’ve applied all three advanced Snowpark patterns:

| Pattern | What You Built |
|---------|---------------|
| Vectorized UDF | Guest health score combining frequency, monetary, and recency |
| Window functions | Cumulative spend, order rank, and 3-order moving average |
| Stored procedure | Order scoring pipeline — callable and schedulable |

### Discussion: NCL Use Cases

How could these patterns apply to NCL’s business?
- **UDFs:** Guest sentiment scoring, cabin pricing formulas, loyalty tier computation
- **Window functions:** Booking sequence analysis, spend trends, onboard activity patterns
- **Stored procedures:** Nightly guest scoring, weekly feature refreshes, automated retraining

In [ ]:
# --- Cleanup ---
try:
    session.sql(f"DROP TABLE IF EXISTS {STUDENT_NAME}_SCORED_ORDERS").collect()
    logger.info("Cleaned up.")
except Exception as e:
    logger.warning("Cleanup note: %s", e)

In [ ]:
session.close()